<a href="https://colab.research.google.com/github/GH-Rumi/PemMes_7_Helmi/blob/main/JS03/JS03_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Import Library
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import chi2, SelectKBest, RFE
from sklearn.metrics import accuracy_score, classification_report

In [10]:
# Load Data
df = pd.read_csv("Titanic-Dataset.csv")

# Pisahkan Survived
y = df["Survived"].astype(int)
X = df.drop(columns=["Survived"])

# Buat list variabel numerik dan kategorikal
# Akan digunakan untuk proses seleksi fitur
# Name tidak akan digunakan karena tidak relevan
num_cols = ["Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Pclass", "Sex", "Embarked"]

Data penumpang Titanic dimuat ke dalam sistem dan dipecah menjadi dua bagian utama: variabel target Survived (disimpan dalam y) dan fitur-fitur prediksi (disimpan dalam X). Selanjutnya, variabel dikelompokkan secara manual menjadi tipe numerik dan kategorikal agar bisa diproses dengan metode yang berbeda pada langkah selanjutnya, sedangkan variabel Name dibuang karena tidak memiliki pola prediksi yang relevan.

In [11]:
# Ekstaksi Fitur dengan Pipeline

# Data Numerik
num_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Data Kategorikal
cat_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

Langkah ini mendefinisikan aturan pembersihan dan transformasi data secara otomatis menggunakan Pipeline. Untuk fitur numerik, nilai yang kosong akan diisi dengan nilai tengah (median) lalu disamakan rentang skalanya. Untuk fitur kategorikal, nilai kosong diisi dengan data yang paling sering muncul (modus), lalu diubah menjadi representasi angka biner menggunakan metode OneHotEncoder.

In [12]:
# Buat Fitur FamilySize
X["FamilySize"] = X["SibSp"].fillna(0) + X["Parch"].fillna(0) + 1

# Tambahkan FamilySize pada kelompok numerikal
preprocess = ColumnTransformer([
    ("num", num_tf, num_cols + ["FamilySize"]),
    ("cat", cat_tf, cat_cols),
])

Anda menciptakan satu variabel baru bernama FamilySize (ukuran keluarga) dengan cara menjumlahkan kolom pasangan/saudara (SibSp), orang tua/anak (Parch), dan 1 (penumpang itu sendiri). Semua instruksi pemrosesan—baik untuk kolom numerik (termasuk fitur baru) maupun kategorikal—kemudian disatukan menggunakan ColumnTransformer.

In [13]:
# Seleksi fitur dengan SelectKBest
# Fungsi tersebut akan menggunakan analisis variance
# Baca: https://scikit-learn.org/stable/modules/feature_selection.html#univariate-feature-selection

from sklearn.feature_selection import f_classif
selector_filter = SelectKBest(score_func=f_classif, k=5)

# Buat pipeline final (INGAT INI HANYA PIPELINE, BELUM MEMPROSES DATA)
pipe_filter = Pipeline([
    ("prep", preprocess), # menjalankan pipeline preprocessing
    ("sel", selector_filter), # menjalankan pipeline seleksi fitur
    ("clf", LogisticRegression(max_iter=1000)) # uji dengan model sederhana -> Logistic Regression
])

Metode SelectKBest dengan fungsi statistik ANOVA disiapkan khusus untuk memilih 5 fitur terbaik yang memiliki hubungan terkuat dengan status keselamatan penumpang. Anda kemudian merangkai Pipeline final (pipe_filter) yang menggabungkan seluruh tahapan dari awal: pemrosesan awal, seleksi 5 fitur, hingga pengujian menggunakan algoritma Logistic Regression.

In [14]:
# Lakukan pelatihan dan uji model
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

pipe_filter.fit(X_train, y_train)
pred = pipe_filter.predict(X_test)
print("=== Filter (ANOVA) + LR ===")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

=== Filter (ANOVA) + LR ===
Accuracy: 0.776536312849162
              precision    recall  f1-score   support

           0       0.80      0.85      0.82       110
           1       0.73      0.67      0.70        69

    accuracy                           0.78       179
   macro avg       0.77      0.76      0.76       179
weighted avg       0.77      0.78      0.77       179



Data keseluruhan dibagi menjadi 80% data untuk melatih model dan 20% data untuk menguji model. Setelah pipeline dijalankan secara penuh, model dievaluasi dan menghasilkan metrik kinerja berupa akurasi sekitar 78%.

In [15]:
# 1) Nama fitur setelah preprocess
feat_names = pipe_filter.named_steps["prep"].get_feature_names_out()
print("Nama fitur:", feat_names)
print("\n")

# 2) Mask & skor fitur terpilih (SelectKBest)
sel = pipe_filter.named_steps["sel"]
mask = sel.get_support()
selected_names = feat_names[mask]
selected_scores = sel.scores_[mask]
top = sorted(zip(selected_names, selected_scores), key=lambda t: t[1], reverse=True)[:10]
print("Top fitur:", top)

Nama fitur: ['num__Age' 'num__SibSp' 'num__Parch' 'num__Fare' 'num__FamilySize'
 'cat__Pclass_1' 'cat__Pclass_2' 'cat__Pclass_3' 'cat__Sex_female'
 'cat__Sex_male' 'cat__Embarked_C' 'cat__Embarked_Q' 'cat__Embarked_S']


Top fitur: [('cat__Sex_female', np.float64(306.5932488951883)), ('cat__Sex_male', np.float64(306.59324889518797)), ('cat__Pclass_3', np.float64(80.33862734392042)), ('cat__Pclass_1', np.float64(73.99727564291717)), ('num__Fare', np.float64(58.31490728198491))]


Langkah terakhir ini digunakan untuk "membedah" pipeline dan membuktikan fitur apa saja yang berhasil lolos seleksi SelectKBest. Hasil ekstraksi menunjukkan bahwa 5 faktor paling krusial yang menentukan akurasi 78% tersebut adalah jenis kelamin (perempuan dan laki-laki), kelas tiket penumpang (kelas 1 dan kelas 3), serta harga tiket (Fare).